<a href="https://colab.research.google.com/github/Rajeeb1819/WebApplication2/blob/main/Python_week1_level1_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mutability —
The Real Mental Model
Don't think "mutable = changeable". Think:

Mutable = the object's internal state can change in-place
Immutable = any "change" creates a new object

python# Immutable — new object created
a = "hello"
id(a)
a += " world"
id(a)  # different id — new object

# Mutable — same object modified
lst = [1, 2, 3]
id(lst)
lst.append(4)
id(lst)  # same id — same object

Why this matters in real code:
pythondef append_item(item, lst=[]):  # Classic trap
    lst.append(item)
    return lst

append_item(1)  # [1]
append_item(2)  # [1, 2]  ← NOT [2]
Default mutable arguments are created once at function definition, not each call. Because lists are mutable and the same object is reused.


###3. Hashing — Why Some Types Can Be Dict Keys
Only immutable (and hashable) types can be dict keys or set members.
pythond = {}
d[(1, 2)] = "ok"     # tuple — hashable ✓
d[[1, 2]] = "fail"   # list — unhashable ✗
Hash must be stable for the object's lifetime. Mutables can't guarantee that, so they're unhashable.
pythonhash("hello")   # consistent
hash([1,2,3])   # TypeError
A custom class is hashable by default (hashed by id). The moment you define __eq__, Python sets __hash__ = None — forcing you to think about it deliberately.

###4. Identity vs Equality
pythona = [1, 2, 3]
b = [1, 2, 3]

a == b   # True  — same value
a is b   # False — different objects

# The trap with small ints and strings (interning)
x = 256
y = 256
x is y   # True  — Python interns small ints (-5 to 256)

x = 257
y = 257
x is y   # False — not interned
Python interns small integers and short strings as an optimization. Never use is to compare values — only to check identity (is None is the one legitimate use).

###5. Memory: References, Not Copies
Variables are labels on objects, not boxes containing values.
pythona = [1, 2, 3]
b = a        # both point to same object

b.append(4)
print(a)     # [1, 2, 3, 4] — a is affected
Shallow vs deep copy:
pythonimport copy

original = [[1, 2], [3, 4]]
shallow = original.copy()       # or original[:]
deep = copy.deepcopy(original)

original[0].append(99)

print(shallow[0])  # [1, 2, 99] — inner list is shared
print(deep[0])     # [1, 2]     — fully independent


##Write a function that demonstrates the mutable default argument bug, then fix it the correct way.
Demonstrating the Mutable Default Argument Bug in Python
In Python, default arguments are evaluated once when the function is defined, rather than every time the function is called. This can lead to unexpected behavior when using mutable objects (like lists or dictionaries) as default arguments, as subsequent calls will share the same object in memory.

1. The Buggy Function
Here is an example demonstrating the issue:

Python
def add_to_list(item, my_list=[]):
    my_list.append(item)
    return my_list

# Demonstration
list1 = add_to_list(1)
print(f"List 1: {list1}")  # Output: [1]

list2 = add_to_list(2)
print(f"List 2: {list2}")  # Output: [1, 2]
Why it happens:
my_list is created just once when the function is defined.

When Notes(2) is called, it reuses the same list instance that was created during the first call, retaining the value 1.

2. The Correct Way to Fix It
To avoid this behavior, the standard practice is to use None as the default value and then create a new list inside the function if the argument is None.

Python
def add_to_list_fixed(item, my_list=None):
    if my_list is None:
        my_list = []
    my_list.append(item)
    return my_list

# Demonstration
list1 = add_to_list_fixed(1)
print(f"List 1: {list1}")  # Output: [1]

list2 = add_to_list_fixed(2)
print(f"List 2: {list2}")  # Output: [2]
Why the fix works:
Every time Notes_fixed is called without providing a list, my_list defaults to None.

The if my_list is None: check ensures that a brand-new, independent list is initialized on every call.

Key Takeaways
Never use mutable objects (like [], {}, or instances of custom classes) as default argument values in Python.

Use None as a sentinel value to indicate that no argument was provided, and initialize the mutable object inside the function body.

##Best Practice: Checking for None
Always use the identity operator (is) rather than the equality operator (==) when checking if a variable is None.



##Problem 2: Create a class Point with x and y. Make it hashable so it can be used as a dict key. Make two Point(1,2) instances equal and have the same hash.
Creating a Hashable Point Class in Python
In Python, for an object to be hashable and usable as a dictionary key, it must meet two requirements:

It must have a __hash__() method.

It must have an __eq__() method.

Its hash value must never change during its lifetime (making the object effectively immutable).

Solution 1: Using the dataclasses Module (Recommended & Pythonic)
The most concise and Pythonic way to achieve this is by using a frozen dataclass. When you set frozen=True, Python automatically makes the instance immutable, generates the __eq__ method, and implements a __hash__ method based on the class attributes.

Python
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: int
    y: int

# Demonstration
p1 = Point(1, 2)
p2 = Point(1, 2)

# 1. Equality check
print(p1 == p2)  # Output: True

# 2. Hash check
print(hash(p1) == hash(p2))  # Output: True

# 3. Using as a dictionary key
my_dict = {p1: "This is a point"}
print(my_dict[p2])  # Output: This is a point
Solution 2: Custom Class Implementation
If you are not using data classes or need fine-grained control over the initialization, you can implement the class manually. To ensure the object's hash doesn't change, we make the attributes immutable using properties.

Python
class Point:
    def __init__(self, x, y):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __eq__(self, other):
        # Check if the other object is an instance of Point and has the same coordinates
        if isinstance(other, Point):
            return self.x == other.x and self.y == other.y
        return False

    def __hash__(self):
        # Create a hash based on an immutable tuple of (x, y)
        return hash((self.x, self.y))

# Demonstration
p1 = Point(1, 2)
p2 = Point(1, 2)

# 1. Equality check
print(p1 == p2)  # Output: True

# 2. Hash check
print(hash(p1) == hash(p2))  # Output: True

# 3. Using as a dictionary key
my_dict = {p1: "Coordinate (1,2)"}
print(my_dict[p2])  # Output: Coordinate (1,2)
Why this works:
__eq__: Compares self.x and self.y values rather than comparing the object's memory addresses.

__hash__: Generates a unique numeric identifier derived from the tuple (x, y). Because the values of x and y are fixed (immutable), the hash value remains consistent throughout the object's lifecycle.



Understanding __eq__ and __hash__ in Python
In Python, __eq__ and __hash__ are two special (dunder) methods that work together to define how objects are compared and stored in hash-based collections like set and dict.

What Are They?
__eq__(self, other): Triggered by the equality operator (==). It defines what makes two objects "equal" in value.

__hash__(self): Returns an integer hash value for the object. This integer is used by data structures like dictionaries and sets to store and retrieve items quickly.

The Golden Rule of Hashing
To prevent inconsistencies, Python enforces a strict rule regarding these two methods:

If two objects are equal (a == b), their hash values must be equal (hash(a) == hash(b)).

However, the reverse is not always true: two objects with the same hash value do not necessarily have to be equal (a scenario known as a hash collision).

How They Work Together
When you look up a key in a dictionary or search a set, Python performs a two-step process:

Hash Lookup: Python calls hash(key) to quickly narrow down the memory location (bucket) where the item should be stored.

Equality Check: If multiple items share the same hash (a collision), Python uses __eq__() to check each item in that bucket until it finds the exact match.

Implementation Example
If you want to create custom hashable objects (for instance, to use them as dictionary keys), you must implement both methods.

Here is how you can implement them manually:

Python
class User:
    def __init__(self, user_id, name):
        self.user_id = user_id
        self.name = name

    def __eq__(self, other):
        if isinstance(other, User):
            return self.user_id == other.user_id
        return False

    def __hash__(self):
        # Hash based on immutable attributes
        return hash(self.user_id)

# Demonstration
u1 = User(101, "Alice")
u2 = User(101, "Bob")

print(u1 == u2)       # Output: True (User IDs are the same)
print(hash(u1) == hash(u2)) # Output: True

# You can now use this object as a dictionary key
my_dict = {u1: "Active"}
print(my_dict[u2])    # Output: Active
Key Takeaways
Mutable objects should not be hashable. If an object's value changes, its hash would change, breaking the dictionary or set data structure. By default, mutable built-in types like list and dict have __hash__ = None.

Use dataclasses.dataclass(frozen=True) when you need a simple hashable object without writing the boilerplate yourself.

Would you like to explore how to explicitly make a custom object unhashable by setting __hash__ = None?